In [2]:
import torch
from typing import List, Dict, Any
from transformers import AutoTokenizer, AutoModelForCausalLM
from vllm import LLM, SamplingParams
from vllm_utils import *
from utils import *
from drgrpo_grader import r1_zero_reward_fn

INFO 03-06 23:34:03 __init__.py:190] Automatically detected platform cuda.


In [3]:
from vllm import LLM, SamplingParams
model_name = '../model/Qwen2.5-Math-1.5B'
vllm = init_vllm(model_name,device='cuda',seed=42,gpu_memory_utilization=0.25)
print('init LLM successfully!')

INFO 03-06 23:34:16 config.py:542] This model supports multiple tasks: {'score', 'classify', 'generate', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 03-06 23:34:16 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='../model/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='../model/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=../model/Qwen2.5-Math-1.5B, num_

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-06 23:34:19 model_runner.py:1115] Loading model weights took 2.8797 GB
INFO 03-06 23:34:21 worker.py:267] Memory profiling takes 1.24 seconds
INFO 03-06 23:34:21 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.25) = 5.90GiB
INFO 03-06 23:34:21 worker.py:267] model weights take 2.88GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 1.56GiB.
INFO 03-06 23:34:21 executor_base.py:110] # CUDA blocks: 3657, # CPU blocks: 9362
INFO 03-06 23:34:21 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 14.29x
INFO 03-06 23:34:27 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utili

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:30<00:00,  1.16it/s]

INFO 03-06 23:34:58 model_runner.py:1562] Graph capturing finished in 30 secs, took 0.20 GiB
INFO 03-06 23:34:58 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 38.02 seconds
init LLM successfully!


In [4]:
def get_ei_batch(
    vllm:LLM,
    sampling_params: SamplingParams,# 温度不应该为1.0,需要有一定随机性
    sampled_questions:List[str],
    sampled_answers:List[str],# only answers,
    rollout_size : int,# 应该为每一个样本采样多少个output
)->tuple[List[str],List[str]]:
    responses = []# List[List[str]]
    for _ in range(rollout_size):
        responses_per_rollout = generate_responses(
            vllm=vllm,
            sampling_params=sampling_params,
            prompts=sampled_questions,
        )
        responses.append(responses_per_rollout)# [['1','2']]

    valid_prompts = []
    valid_ground_truths = []

    for i in range(rollout_size):
        for p,o,ans in zip(sampled_questions,responses[i],sampled_answers):
            reward_dict = r1_zero_reward_fn(o,ans,fast=True)
            if reward_dict['reward'] == 1.0:
                valid_prompts.append(p)
                valid_ground_truths.append(o)
    return responses, valid_prompts, valid_ground_truths

In [5]:
# load_template
template_path = '../cs336_alignment/prompts/r1_zero.prompt'
r1_template = load_template(template_path)

# generation
import os 
json_path  = '../preprocessed/math/test.jsonl'
prompts = get_r1_prompts(json_path,r1_template)
ground_truths = get_r1_ground_truths(json_path)# only answer
ground_truths_with_template = get_r1_ground_truths_with_template(json_path)#) # with template

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=1024,
    stop=["</answer>"],
    include_stop_str_in_output = True # in need
)
prompts[1],ground_truths[1],ground_truths_with_template[1]

('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Define\n\\[p = \\sum_{k = 1}^\\infty \\frac{1}{k^2} \\quad \\text{and} \\quad q = \\sum_{k = 1}^\\infty \\frac{1}{k^3}.\\]Find a way to write\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3}\\]in terms of $p$ and $q.$\nAssistant: <think>',
 'p - q',
 'We count the number of times $\\frac{1}{n^3}$ appears in the sum\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3},\\]where $n$ is a fixed positive integer.  (In other words, we are conditioning the sum on $j + k$.)  We get a term of $\\frac{1}{n^3}$ each time $j + k = n.$  The pairs $

In [5]:
evaluate_vllm(
    vllm=vllm,
    prompts=prompts,
    ground_truths=ground_truths,
    sampling_params=sampling_params,
    reward_fn=r1_zero_reward_fn,
)

Processed prompts:  27%|██▋       | 135/500 [00:15<00:35, 10.39it/s, est. speed input: 1377.84 toks/s, output: 988.66 toks/s]

WARNING 03-06 22:54:12 scheduler.py:1560] Sequence group 393 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1


Processed prompts:  50%|████▉     | 248/500 [00:23<00:18, 13.43it/s, est. speed input: 1642.48 toks/s, output: 1497.62 toks/s]

WARNING 03-06 22:54:20 scheduler.py:1560] Sequence group 433 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=51


Processed prompts:  63%|██████▎   | 317/500 [00:31<00:20,  8.95it/s, est. speed input: 1603.18 toks/s, output: 1720.20 toks/s]

WARNING 03-06 22:54:28 scheduler.py:1560] Sequence group 451 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=101


Processed prompts:  73%|███████▎  | 365/500 [00:39<00:16,  8.08it/s, est. speed input: 1463.46 toks/s, output: 1851.96 toks/s]

WARNING 03-06 22:54:36 scheduler.py:1560] Sequence group 446 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=151


Processed prompts: 100%|██████████| 500/500 [01:02<00:00,  7.99it/s, est. speed input: 1311.37 toks/s, output: 2782.86 toks/s]


{'sample_size': 500,
 'answer_correct': 46,
 'format_correct': 137,
 'total_correct': 46,
 'format_correct_but_answer_wrong': 91,
 'answer_correct_but_format_wrong': 0,
 'total_wrong': 363,
 'accuracy': 0.092,
 'wrong_rate': 0.726,
 'contradictory_samples': 0}

In [7]:
responses, valid_prompts, valid_ground_truths = get_ei_batch(
    vllm=vllm,
    sampling_params=sampling_params,
    sampled_questions=prompts,
    sampled_answers=ground_truths,
    rollout_size=4
)

Processed prompts:  27%|██▋       | 135/500 [00:16<00:51,  7.08it/s, est. speed input: 1330.83 toks/s, output: 954.93 toks/s]

WARNING 03-06 23:35:16 scheduler.py:1560] Sequence group 393 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1


Processed prompts:  50%|████▉     | 248/500 [00:24<00:21, 11.72it/s, est. speed input: 1572.64 toks/s, output: 1433.98 toks/s]

WARNING 03-06 23:35:24 scheduler.py:1560] Sequence group 433 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=51


Processed prompts:  63%|██████▎   | 317/500 [00:32<00:27,  6.77it/s, est. speed input: 1522.25 toks/s, output: 1633.36 toks/s]

...

Processed prompts:  63%|██████▎   | 317/500 [00:37<00:17, 10.59it/s, est. speed input: 1299.00 toks/s, output: 1502.60 toks/s]

WARNING 03-06 23:39:00 scheduler.py:1560] Sequence group 1942 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=801


Processed prompts:  70%|███████   | 350/500 [00:42<00:22,  6.61it/s, est. speed input: 1279.23 toks/s, output: 1598.03 toks/s]

WARNING 03-06 23:39:06 scheduler.py:1560] Sequence group 1944 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=851


Processed prompts: 100%|██████████| 500/500 [01:06<00:00,  7.50it/s, est. speed input: 1231.51 toks/s, output: 2657.66 toks/s]


In [8]:
len(valid_prompts),len(valid_ground_truths)

(158, 158)

In [9]:
valid_prompts[0], valid_ground_truths[0]

('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: How many positive whole-number divisors does 196 have?\nAssistant: <think>',
 ' The number 196 can be factored into its prime factors as 2^2 * 7^2. To find the number of positive whole-number divisors, we use the formula for finding the total number of divisors of a number given its prime factorization. If a number N has the prime factorization p1^a * p2^b * ... * pk^k, then the total number of divisors is (a+1)(b+1)...(k+1). Applying this to 196, we have (2+1)(2+1) = 3 * 3 = 9. Therefore, 196 has 9 positive whole-number divisors.</think> <answer> 9 </answer>')

# EI Dataset
+ 包含SFTDataset(完整的training set)
+ 采样方法，返回一个完整的SFTDataset

In [11]:
from sft import *
from sft_trainer import SFTDataset
from torch.utils.data import DataLoader,Dataset

In [ ]:
class EIDataset(Dataset):
    def __init__(
        self,
        json_path:str,# only training set
        prompt_template_path:str,
        rollout_size:int = 8,
    ):
        self.trainingset = SFTDataset(json_path,prompt_template_path)
        self.rollout_size = rollout_size
        
    def __len__(self):
        return len(self.trainingset)

    def __getitem__(self, index):
        return self.trainingset[index]
    
    @torch.no_grad()
    def sample_responses(self)->tuple[List[str],List[str]]:
        assert len(self.trainingset.prompts) == len(self.trainingset.answers), \
    f"数据集长度不匹配: prompts {len(self.trainingset.prompts)} vs answers {len(self.trainingset.answers)}" 
        indices = range(len(self.trainingset.prompts))
        sampled_indices = random.sample(indices,self.rollout_size)
        sampled_prompts = [self.trainingset.prompts[idx] for idx in sampled_indices]
        sampled_answers = [self.trainingset.answers[idx] for idx in sampled_indices]
        return sampled_prompts,sampled_answers
    
    def get_ei_batch(
        self,
        vllm:LLM,
        sampling_params: SamplingParams,# 温度不应该为1.0,需要有一定随机性
        sampled_questions:List[str],
        sampled_answers:List[str],# only answers,
        rollout_size : int,# 应该为每一个样本采样多少个output
    )->tuple[List[str],List[str]]:
        responses = []# List[List[str]]
        for _ in range(rollout_size):
            responses_per_rollout = generate_responses(
                vllm=vllm,
                sampling_params=sampling_params,
                prompts=sampled_questions,
            )
            responses.append(responses_per_rollout)# [['1','2']]

        valid_prompts = []
        valid_ground_truths = []

        for i in range(rollout_size):
            for p,o,ans in zip(sampled_questions,responses[i],sampled_answers):
                reward_dict = r1_zero_reward_fn(o,ans,fast=True)
                if reward_dict['reward'] == 1.0:
                    valid_prompts.append(p)
                    valid_ground_truths.append(o)
        # valid_prompts, valid_ground_truths 可能为空
        return responses, valid_prompts, valid_ground_truths
        

In [39]:
ei_dataset = EIDataset(
    json_path = '../preprocessed/math/train.jsonl',
    prompt_template_path = '../cs336_alignment/prompts/r1_zero.prompt',
    rollout_size=8
)
prompts, answers = ei_dataset.sample_responses()
prompts[0], answers[0]

('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: In my neighborhood, there are six streets. There are 10 houses on each side of each street. No house faces two different streets. How many houses are in my neighborhood?\nAssistant: <think>',
 '120')

In [40]:
responses,valid_prompts,valid_ground_truths = ei_dataset.get_ei_batch(
    vllm=vllm,
    sampling_params=sampling_params,
    sampled_questions=prompts,
    sampled_answers=answers,
    rollout_size=2
)


Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 8/8 [00:05<00:00,  1.39it/s, est. speed input: 202.08 toks/s, output: 363.92 toks/s]


In [ ]:
len(responses),len(responses[0])

(2, 8)

: 